# Proyecto Final: Herramientas para la Inteligencia Artificial

Este cuaderno de Jupyter representa el proyecto final de la asignatura 'Herramientas para la Inteligencia Artificial'. En él se describen, paso a paso, las tareas realizadas para el cumplimiento de los requerimientos especificados por el docente.

Como punto de partida requerimos instalar las siguientes librerías los cuales son fundamentales para la ejecución de nuestro código.
* **Pandas :** Librería utilizado para la lectura y modificación de nuestra
fuentes de datos
* **Bokeh :** Librería utilizado para mostrar los gráficos de barras


In [29]:
#!pip install pandas
#!pip install bokeh

### Importación de Librerías

Se deben importar las siguientes librerías para el desarrollo del proyecto:

- **`sqlite3`**: Cabe mencionar que esta librería ya viene integrada en el entorno estándar de Python, por lo que no requiere instalación previa. Será utilizada para la creación y gestión de nuestra base de datos local, permitiendo el almacenamiento de una de nuestras fuentes de datos.
- **`pandas`**: Para lectura y escritura de archivos
- **`matplotlib`**: Se importan todas las dependencias necesarias de este modulo.


In [32]:
import sqlite3 as sqlite
import pandas as pd
import os
from bokeh.io import output_file,output_notebook,output_notebook, show
from bokeh.models import ColumnDataSource,DataTable,TableColumn
from bokeh.layouts import column
from bokeh.plotting import figure

### Fuentes de Datos Seleccionadas

Para cumplir con los requerimientos de este proyecto, se investigó y decidió utilizar dos fuentes de datos principales:

1. **Fuente de datos de turismo:** Contiene información detallada sobre diversos países, volumen de visitas por año y ganancias generadas, entre otros indicadores relevantes. Este dataset fue obtenido de Kaggle y se encuentra disponible en el siguiente enlace: [Enlace a Kaggle](https://www.kaggle.com/datasets/umeradnaan/tourism-dataset?resource=download).
2. **Fuente de datos climáticos:** Este dataset recopila registros meteorológicos de múltiples países, incluyendo fechas de actualización, país de origen, humedad, temperatura y otras variables clave para el análisis solicitado. Esta información se extrajo de la plataforma Kaggle mediante el siguiente enlace: [Enlace a Kaggle](https://www.kaggle.com/datasets/nelgiriyewithana/global-weather-repository/#site-content).

### Creación de la Base de Datos y Carga de Información

Para cumplir con los requerimientos del proyecto ante la ausencia de una base de datos previa, se procedió a crear una base de datos local utilizando las herramientas que facilita la librería `pandas`. En esta base de datos se alojará una tabla denominada `datos`.

Para llevar a cabo este proceso, se definieron las rutas (*paths*) de almacenamiento donde se guardará la base de datos y desde donde se leerá el archivo original de clima. El flujo consiste en cargar los datos provenientes del archivo `.csv` directamente hacia la base de datos estructurada.

In [33]:
from google.colab import drive
drive.mount('/content/drive')

weather_db = "/content/drive/MyDrive/ToolsAIProject/weather/wather.db"
weather_csv = "/content/drive/MyDrive/ToolsAIProject/weather/GlobalWeatherRepository.csv"


conn = sqlite.connect(weather_db)

weather_df = pd.read_csv(weather_csv)

weather_df.to_sql("datos",conn,if_exists="replace",index=False)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


143262

### Consultas a la Base de Datos con Pandas

Una vez culminada la carga de datos, es posible acceder y realizar consultas estructuradas a la base de datos `SQLite`. Para este propósito, se vuelven a utilizar los métodos que nos facilita la librería `pandas`, los cuales optimizan y simplifican las tareas de extracción y manipulación de información directamente en DataFrames.

In [34]:
query = "SELECT * FROM datos"

weathers = pd.read_sql(query, conn)

print(weathers.shape)
print(weathers.columns)

(143262, 41)
Index(['country', 'location_name', 'latitude', 'longitude', 'timezone',
       'last_updated_epoch', 'last_updated', 'temperature_celsius',
       'temperature_fahrenheit', 'condition_text', 'wind_mph', 'wind_kph',
       'wind_degree', 'wind_direction', 'pressure_mb', 'pressure_in',
       'precip_mm', 'precip_in', 'humidity', 'cloud', 'feels_like_celsius',
       'feels_like_fahrenheit', 'visibility_km', 'visibility_miles',
       'uv_index', 'gust_mph', 'gust_kph', 'air_quality_Carbon_Monoxide',
       'air_quality_Ozone', 'air_quality_Nitrogen_dioxide',
       'air_quality_Sulphur_dioxide', 'air_quality_PM2.5', 'air_quality_PM10',
       'air_quality_us-epa-index', 'air_quality_gb-defra-index', 'sunrise',
       'sunset', 'moonrise', 'moonset', 'moon_phase', 'moon_illumination'],
      dtype='object')


### Preparación y Limpieza de Datos para la Unión

Con el fin de cumplir con los criterios de aceptación, se procedió a identificar los identificadores (IDs) o variables clave que permitan realizar la unión (*merge*) de los conjuntos de datos. Para este proyecto, se determinó que las columnas de enlace serán `country` del dataset de climas y `Country` del dataset de turismo.

Antes de ejecutar la unión, se realizó una depuración para eliminar filas duplicadas utilizando la columna `country` como criterio. Finalmente, se imprimieron las primeras 2 filas del conjunto de datos resultante para validar que la modificación se haya implementado correctamente.

In [35]:
columns_to_print=["country","timezone","temperature_celsius","last_updated"]

# df_city = weathers[weathers["country"]=="China"]
# print(df_city[columns_to_print].head(5))

weather_without_duplicated_df = weathers.drop_duplicates(subset="country")
print(weather_without_duplicated_df[columns_to_print].head(2))

       country       timezone  temperature_celsius      last_updated
0  Afghanistan     Asia/Kabul                 26.6  2024-05-16 13:15
1      Albania  Europe/Tirane                 19.0  2024-05-16 10:45


### Lectura de la Segunda Fuente de Datos

A continuación, se procedió a realizar la lectura de la segunda fuente de datos. A diferencia del dataset anterior, este conjunto de información no requiere de procesos adicionales de almacenamiento o gestión en la base de datos local. Asimismo, para los fines de este análisis, no es un requisito que el campo `Country` contenga valores únicos en esta etapa.

In [36]:
tourism_df = pd.read_csv("/content/drive/MyDrive/ToolsAIProject/tourism_dataset.csv")

print(tourism_df.shape)
print(tourism_df.columns)

print(tourism_df['Country'].is_unique)

(5989, 7)
Index(['Location', 'Country', 'Category', 'Visitors', 'Rating', 'Revenue',
       'Accommodation_Available'],
      dtype='object')
False


### Integración de Fuentes de Datos (*Merge*)

Se realizó la unión de ambos conjuntos de datos utilizando el método `merge` de la librería `pandas`. En los parámetros de la función, se especificó en primer lugar el dataset climáticos y, en segundo lugar, el dataset de turismo; posteriormente, se definieron las columnas clave que el algoritmo debe considerar para efectuar el acoplamiento. Como resultado de esta operación, se obtuvo un nuevo `DataFrame` unificado.

In [37]:
result_merge_df = pd.merge(
    weather_without_duplicated_df,
    tourism_df,
    left_on="country",
    right_on="Country",
    how="inner"
)
print(result_merge_df.shape)
print(result_merge_df.head(2))
print(result_merge_df.columns)


(5141, 48)
     country location_name  latitude  longitude          timezone  \
0  Australia      Canberra    -35.28     149.22  Australia/Sydney   
1  Australia      Canberra    -35.28     149.22  Australia/Sydney   

   last_updated_epoch      last_updated  temperature_celsius  \
0          1715849100  2024-05-16 18:45                  9.0   
1          1715849100  2024-05-16 18:45                  9.0   

   temperature_fahrenheit condition_text  ...     moonset      moon_phase  \
0                    48.2          Clear  ...  No moonset  Waxing Gibbous   
1                    48.2          Clear  ...  No moonset  Waxing Gibbous   

   moon_illumination    Location    Country  Category  Visitors  Rating  \
0                 55  OcCopAsiyJ  Australia  Cultural     59309     4.1   
1                 55  dUCLjskBYA  Australia     Urban     25568     4.2   

     Revenue  Accommodation_Available  
0  588988.55                      Yes  
1  941320.46                      Yes  

[2 rows x

### Visualización de Datos con Bokeh

A partir de este punto, se dispone de una única fuente de datos consolidada mediante la operación de acoplamiento (*merge*). En esta sección, se procederá a realizar la visualización de los datos aplicando gráficos bajo criterios personales.

Como primer elemento visual, se presenta una tabla interactiva que consolida la lista de datos turísticos junto con las variables climáticas, implementada mediante la librería `Bokeh`.

> **Nota para Google Colab:** Debido a que este proyecto se ejecuta en el entorno de Google Colab, es indispensable inicializar el método `output_notebook()` de Bokeh antes de renderizar cualquier elemento para asegurar la correcta visualización de los gráficos en el cuaderno.

In [41]:
output_notebook()
source = ColumnDataSource(result_merge_df)
columns = [

    TableColumn(field="Visitors", title="jj", width=400),
    TableColumn(field="Rating", title="Rating", width=80),
    TableColumn(field="Revenue", title="Ingresos", width=120),
    TableColumn(field="temperature_celsius", title="Temperatura", width=120)
]

data_table = DataTable(source=source, columns=columns, width=800, height=400)

# output_file("tabla_datos.html")

show(data_table)

### Análisis de Visitantes por País (Histograma)

Para la segunda representación visual, se analizó la distribución de países en función de su número de visitantes. Con el fin de cumplir con este requerimiento, se procedió a crear un nuevo `DataFrame` mediante el agrupamiento (*group by*) de los datos por la variable país, aplicando una función de agregación para sumar el total de visitantes de cada uno. Posteriormente, esta información consolidada se proyectó en el gráfico correspondiente.

In [43]:

visitors_by_country = result_merge_df.groupby('Country')['Visitors'].sum().reset_index()

source_pais = ColumnDataSource(visitors_by_country)

p = figure(x_range=visitors_by_country['Country'],
           title="Visitantes por país",
           x_axis_label="Ubicación",
           y_axis_label="Visitantes",
           height=600,
           width=600)


p.vbar(x='Country', top='Visitors', source=source_pais,
       width=0.5, color='steelblue', legend_label="Visitantes")


p.xaxis.major_label_orientation = 65
p.legend.location = "top_left"


show(p)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Análisis de Temperatura por País

Por último, se realiza una visualización para comparar las temperaturas registradas en los diferentes países.

In [44]:
temp_by_country = result_merge_df.groupby('Country')['temperature_celsius'].mean().reset_index()
temp_by_country = temp_by_country.sort_values('temperature_celsius', ascending=False)

print("Temperatura promedio por país:")
print(temp_by_country)

source = ColumnDataSource(temp_by_country)

p = figure(x_range=temp_by_country['Country'],
           title="Temperatura Promedio por País",
           x_axis_label="País",
           y_axis_label="Temperatura Promedio (°C)",
           height=400,
           width=700,
           )

p.vbar(x='Country', top='temperature_celsius', source=source,
       width=0.5, color='coral', alpha=0.8)

p.xaxis.major_label_orientation = 45

show(p)

Temperatura promedio por país:
     Country  temperature_celsius
5      India                 42.0
2      China                 29.0
3      Egypt                 27.0
1     Brazil                 23.1
4     France                 15.0
0  Australia                  9.0



Estos últimos bloques de código corresponden a líneas que generan el csv final utilizado y el sql de la base de datos.

In [13]:
# turism_weather_csv = "/content/drive/MyDrive/ToolsAIProject/weather/turism_weather.csv"

# result_merge_df.to_csv(turism_weather_csv,index=False)

In [14]:

# def export_db_to_sql(archivo_db, archivo_sql='backup.sql'):

#     with open(archivo_sql, 'w', encoding='utf-8') as f:
#         for linea in conn.iterdump():
#             f.write(f'{linea}\n')

#     conn.close()
#     import os
#     tamaño = os.path.getsize(archivo_sql)
#     print(f"Base de datos exportada: {archivo_sql}")
#     print(f"Tamaño: {tamaño:,} bytes")
#     print(f"Puedes restaurar con: sqlite3 nueva.db < {archivo_sql}")

#     return archivo_sql

# export_db_to_sql('mi_base_datos_completa.sql')